In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    explained_variance_score
)
import optuna
from sklearn.model_selection import cross_val_score

from scipy.stats import pearsonr
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline



d:\Miniconda3\envs\PyTorchVirtualEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
print("hello")

hello


In [3]:
data_df = pd.read_csv("D:\APSirCNTandLSTM\CNT with LSTM\Datasets\CNT_Reinforced Concrete_Final Dataset.csv") # Read the data

data_df.dropna(inplace = True) # Drop missing/null values

print("Original Data Shape: ", data_df.shape)

def is_numeric_row(row):
    try:
        pd.to_numeric(row, errors='raise')
        return True
    except:
        return False
    

numeric_rows = data_df.apply(lambda row: is_numeric_row(row), axis=1)
data_df = data_df[numeric_rows]

print("Data Shape (rows, cols): ", data_df.shape) # Print data shape
data_df # Print/visualize the first 5 rows of the data

Original Data Shape:  (295, 7)
Data Shape (rows, cols):  (295, 7)


,Cement (kg/m³),w/c,Coarse aggregate (kg/m³),Fine aggregate (kg/m³),Curing (days),CNT (%),Compressive strength(MPa)
0,360.0,0.50,950.0,745.0,28,0.00,39.500
1,320.0,0.56,898.0,782.0,28,0.00,40.000
2,328.5,0.68,533.7,625.1,28,0.00,40.000
3,360.0,0.50,898.0,797.0,28,0.00,40.000
4,320.0,0.56,845.0,834.0,28,0.00,40.300
...,...,...,...,...,...,...,...
290,400.0,0.45,987.0,780.0,28,0.05,64.000
291,450.0,0.35,1006.0,768.0,28,0.05,84.900
292,525.0,0.45,0.0,1580.0,28,0.00,29.875
293,525.0,0.45,0.0,1580.0,28,0.06,38.750


In [4]:
#Converting non-numeric columns to numeric, if any. This will convert non-numeric values to NaN, which will be dropped later.

for column in data_df.columns:
    data_df[column] = pd.to_numeric(data_df[column], errors='coerce')


target_column = "Compressive strength(MPa)"

# Check data types
print("Data types:")
print(data_df.dtypes)
print("\n" + "="*50 + "\n")

# Check if target column has issues
print(f"Target column '{target_column}' unique values (first 10):")
print(data_df[target_column].unique()[:10])
print(f"\nTarget column type: {data_df[target_column].dtype}")

# Check for any remaining object columns
object_cols = data_df.select_dtypes(include=['object']).columns.tolist()
print(f"\nObject (string) columns: {object_cols}")

Data types:
Cement (kg/m³)               float64
w/c                          float64
Coarse aggregate (kg/m³)     float64
Fine aggregate (kg/m³)       float64
Curing (days)                  int64
CNT (%)                      float64
Compressive strength(MPa)    float64
dtype: object


Target column 'Compressive strength(MPa)' unique values (first 10):
[39.5 40.  40.3 40.8 41.  41.1 41.3 41.5 41.8 42. ]

Target column type: float64

Object (string) columns: []


In [5]:
# Prepare Input and Output

X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)


Input Shape: (295, 6)
Output Shape: (295,)


In [6]:
X, Y

(array([[3.600e+02, 5.000e-01, 9.500e+02, 7.450e+02, 2.800e+01, 0.000e+00],
        [3.200e+02, 5.600e-01, 8.980e+02, 7.820e+02, 2.800e+01, 0.000e+00],
        [3.285e+02, 6.800e-01, 5.337e+02, 6.251e+02, 2.800e+01, 0.000e+00],
        ...,
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 0.000e+00],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 6.000e-02],
        [5.250e+02, 4.500e-01, 0.000e+00, 1.580e+03, 2.800e+01, 1.200e-01]]),
 array([39.5  , 40.   , 40.   , 40.   , 40.3  , 40.8  , 40.8  , 41.   ,
        41.1  , 41.3  , 41.3  , 41.5  , 41.5  , 41.5  , 41.5  , 41.8  ,
        42.   , 42.1  , 42.4  , 42.5  , 43.   , 43.2  , 43.3  , 43.5  ,
        43.6  , 43.8  , 44.   , 44.5  , 45.   , 45.   , 45.1  , 45.7  ,
        46.   , 46.   , 47.1  , 47.7  , 50.   , 36.8  , 43.1  , 38.6  ,
        47.2  , 45.1  , 49.6  , 47.4  , 54.   , 50.1  , 37.8  , 44.1  ,
        40.9  , 47.5  , 45.3  , 42.5  , 52.   , 48.7  , 46.6  , 54.5  ,
        53.1  , 49.2  , 4

In [7]:

# 80 : 20

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.20, random_state=42
)

print("Training X:", X_train.shape)
print("Testing X:", X_test.shape)

print("Training y:", y_train.shape)
print("Testing y:", y_test.shape)


print("Training set is: ", X_train.shape[0], " rows which is ", round(X_train.shape[0]/data_df.shape[0],4)*100, "%") # Print training shape
print("Testing set is: ",X_test.shape[0], " rows which is ", round(X_test.shape[0]/data_df.shape[0],4)*100, "%") # Print testing shape


Training X: (236, 6)
Testing X: (59, 6)
Training y: (236,)
Testing y: (59,)
Training set is:  236  rows which is  80.0 %
Testing set is:  59  rows which is  20.0 %


In [8]:
scaler_X_XGB = StandardScaler()

X_train_scaled = scaler_X_XGB.fit_transform(X_train)
X_test_scaled = scaler_X_XGB.transform(X_test)

print("Training Shape X:", X_train_scaled.shape)
print("testing Shape X:", X_test_scaled.shape)
print("**********************************")
print("Training Shape y:", y_train.shape)
print("testing Shape y:", y_test.shape)


Training Shape X: (236, 6)
testing Shape X: (59, 6)
**********************************
Training Shape y: (236,)
testing Shape y: (59,)


In [9]:
# Model

svr_model = SVR(
    kernel='rbf',
    C=100,
    epsilon=0.1,
    gamma='scale'
)


# Training
svr_model.fit(
    X_train_scaled,
    y_train
)

# Prediction
y_pred_train = svr_model.predict(X_train_scaled)

y_pred_test = svr_model.predict(X_test_scaled)

print(y_pred_test)
print(y_pred_train)

[55.35022566 56.83959707 53.17811499 49.90701242 37.97327191 38.45498318
 51.13637465 47.60704731 52.01219854 53.29973056 53.36913289 31.3376127
 47.74623223 51.51006897 53.84311561 39.33788451 51.5076405  42.33619116
 37.71783005 52.54109438 59.50012921 59.25557484 53.08433302 58.59993577
 41.81901582 39.24430597 42.71188948 19.77225036 43.99989393 41.44430973
 35.0070186  46.7724397  38.05888264 38.05888264 35.59553786 40.00664874
 60.92217003 55.38580922 47.58051404 31.95652459 28.92415899 52.64606827
 49.69410179 50.70058182 45.36807653 40.94019057 45.90533489 42.94321522
 40.92061616 49.03884089 48.82516414 59.06258705 40.95977208 40.1592703
 16.26205817 42.25922369 51.5076405  48.36341317 55.38580922]
[46.70039779 38.09484111 42.53295876 43.0791824  37.71783005 59.60011078
 48.67533598 53.98796115 49.2968125  40.00664874 54.90036086 59.50012921
 50.27511853 40.82648363 50.24140896 44.99446569 56.83023345 36.70023881
 40.97297748 41.39995953 56.32246478 45.32457361 58.40031154 39.

In [10]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    # avoid divide-by-zero
    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    metrics = {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }

    return metrics

print("TRAIN METRICS")
print(evaluate_regression(y_train, y_pred_train))


print("\nTEST METRICS")
print(evaluate_regression(y_test, y_pred_test))

TRAIN METRICS
{'R2': 0.843149321989552, 'MSE': 16.90937382277818, 'RMSE': 4.112100901337196, 'MAE': 2.062381120027492, 'MAPE (%)': 5.5896284105327405, 'Explained Variance Score': 0.8432097517629084, 'Pearson Correlation': 0.9197630049878792, 'Relative RMSE (%)': 9.0089709737844}

TEST METRICS
{'R2': 0.7471237079430952, 'MSE': 36.70485065289113, 'RMSE': 6.058452826662195, 'MAE': 3.4813687483265747, 'MAPE (%)': 12.179159562596407, 'Explained Variance Score': 0.7623173828550746, 'Pearson Correlation': 0.8805654314292526, 'Relative RMSE (%)': 13.705698046919304}


In [11]:
import pandas as pd

train_metrics = evaluate_regression(y_train, y_pred_train)
test_metrics  = evaluate_regression(y_test, y_pred_test)

df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})

print(df_results.round(4))

                            Train     Test
R2                         0.8431   0.7471
MSE                       16.9094  36.7049
RMSE                       4.1121   6.0585
MAE                        2.0624   3.4814
MAPE (%)                   5.5896  12.1792
Explained Variance Score   0.8432   0.7623
Pearson Correlation        0.9198   0.8806
Relative RMSE (%)          9.0090  13.7057


In [12]:

# OPTUNA + XGBOOST + 5-FOLD CV

# K-Fold

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Objective function
def objective(trial):

    kernel = trial.suggest_categorical(
        "kernel",
        ["rbf", "poly", "sigmoid"]
    )

    params = {

        "C": trial.suggest_float(
            "C",
            1e-2,
            1e3,
            log=True
        ),

        "epsilon": trial.suggest_float(
            "epsilon",
            0.001,
            1.0,
            log=True
        ),

        "gamma": trial.suggest_categorical(
            "gamma",
            ["scale", "auto"]
        ),

        "kernel": kernel
    }

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(**params))
    ])

    scores = cross_val_score(
        pipeline,
        X_train,      # unscaled data
        y_train,
        cv=kfold,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()


In [13]:
study = optuna.create_study(
    direction="maximize",
    study_name="svr_study",
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-03 16:04:55,363] A new study created in memory with name: svr_study
[I 2026-06-03 16:04:57,665] Trial 0 finished with value: -847.9913838566088 and parameters: {'kernel': 'sigmoid', 'C': 100.94996063160931, 'epsilon': 0.3152025039318807, 'gamma': 'auto'}. Best is trial 0 with value: -847.9913838566088.
[I 2026-06-03 16:04:59,355] Trial 1 finished with value: -10.412222561108837 and parameters: {'kernel': 'sigmoid', 'C': 14.154123914338477, 'epsilon': 0.006316151247676288, 'gamma': 'auto'}. Best is trial 1 with value: -10.412222561108837.
[I 2026-06-03 16:05:01,119] Trial 2 finished with value: 0.3443595820180122 and parameters: {'kernel': 'poly', 'C': 1.2261346132188762, 'epsilon': 0.7930566652973096, 'gamma': 'auto'}. Best is trial 2 with value: 0.3443595820180122.
[I 2026-06-03 16:05:02,566] Trial 3 finished with value: 0.2688791785522847 and parameters: {'kernel': 'sigmoid', 'C': 1.7415822858535919, 'epsilon': 0.49318939260656613, 'gamma': 'auto'}. Best is trial 2 with va

In [14]:
print("Best Parameters:")
print(study.best_params)

print("\nBest CV R2:")
print(study.best_value)

Best Parameters:
{'kernel': 'rbf', 'C': 635.1601104588483, 'epsilon': 0.12635834744289212, 'gamma': 'auto'}

Best CV R2:
0.7546565689509355


In [15]:
best_svr = SVR(
    **study.best_params
)

best_svr.fit(
    X_train_scaled,
    y_train
)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'auto'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",635.1601104588483
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.12635834744289212
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [16]:
y_pred_train = best_svr.predict(X_train_scaled)
y_pred_test  = best_svr.predict(X_test_scaled)

In [18]:
def evaluate_regression(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + 1e-8))
    ) * 100

    evs = explained_variance_score(y_true, y_pred)

    pearson_r, _ = pearsonr(y_true, y_pred)

    relative_rmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance Score": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": relative_rmse
    }


train_metrics = evaluate_regression(y_train, y_pred_train)


test_metrics = evaluate_regression(y_test, y_pred_test)



df_results = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics
})


print(df_results.round(4))



                            Train     Test
R2                         0.9059   0.7583
MSE                       10.1404  35.0798
RMSE                       3.1844   5.9228
MAE                        1.5673   3.5317
MAPE (%)                   3.8992  11.3706
Explained Variance Score   0.9066   0.7655
Pearson Correlation        0.9525   0.8749
Relative RMSE (%)          6.9765  13.3989


In [20]:
#Save XGBoost model using pickle

import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\SVR\Saved models\SVR_model8020.pkl", "wb") as f:
    pickle.dump(best_svr, f)

print("SVR model saved successfully.")

SVR model saved successfully.


In [22]:
import pickle

# save trained model
with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\SVR\Saved models\SVR_scalar8020.pkl", "wb") as f:
    pickle.dump(scaler_X_XGB, f)

print("SVR scalar saved successfully.")

SVR scalar saved successfully.


In [23]:
#Load model back

import pickle

with open("D:\APSirCNTandLSTM\CNT with LSTM\practice_Finalized For LSTM-CNT Scripts\practice_LSTM and ANN_for publication\LSTM Improved\EnsemblesANNLSTM8020\SVR\Saved models\SVR_model8020.pkl", "rb") as f:
    loaded_svr = pickle.load(f)

print("Model loaded successfully.")

Model loaded successfully.


In [24]:
#Recompute predictions (train / val / test)

y_pred_train_loaded = loaded_svr.predict(X_train_scaled)
y_pred_test_loaded  = loaded_svr.predict(X_test_scaled)

In [26]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    explained_variance_score
)

from scipy.stats import pearsonr


def evaluate(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs = explained_variance_score(y_true, y_pred)
    pearson_r, _ = pearsonr(y_true, y_pred)
    rrmse = (rmse / np.mean(y_true)) * 100

    return {
        "R2": r2,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "Explained Variance": evs,
        "Pearson Correlation": pearson_r,
        "Relative RMSE (%)": rrmse
    }

train_metrics_loaded = evaluate(y_train, y_pred_train_loaded)
test_metrics_loaded  = evaluate(y_test, y_pred_test_loaded)


results_df = pd.DataFrame({
    "Train": train_metrics_loaded,
    "Test": test_metrics_loaded
})

print(results_df.round(4))

                       Train     Test
R2                    0.9059   0.7583
MSE                  10.1404  35.0798
RMSE                  3.1844   5.9228
MAE                   1.5673   3.5317
MAPE (%)              3.8992  11.3706
Explained Variance    0.9066   0.7655
Pearson Correlation   0.9525   0.8749
Relative RMSE (%)     6.9765  13.3989
